**Cell 1: Import Libraries and Load CC Calls Data**  
This cell imports the required libraries, loads the merged dataset, and selects only the CC calls-related columns plus the churn outcome.  
- **Purpose**: Prepare the data and notebook environment for CC calls hypothesis testing.  
- **Key libraries**: `pandas`, `scipy.stats`, `statsmodels.stats.multitest`.  
- **No hypothesis testing here**: this is data setup only.  
- **Expected output**: a preview of the CC calls data.

In [15]:
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../../data/04_merged/final_merged_features.csv")

# Select only cc_calls related columns
cc_calls_cols = [
    "co_ref", "has_cc_calls_data", "total_cc_calls", "last_cc_call", "cutoff_date_cc_calls",
    "days_since_last_cc_call", "cc_calls_last_7", "cc_calls_last_30", "total_complaints",
    "customer_issues", "financial_issues", "pricing_mentions", "avg_cc_sentiment",
    "repeat_call_ratio", "high_call_volume", "prospect_outcome"
]

df = df[cc_calls_cols].dropna(subset=["prospect_outcome"])

df.head()

,co_ref,has_cc_calls_data,total_cc_calls,last_cc_call,cutoff_date_cc_calls,days_since_last_cc_call,cc_calls_last_7,cc_calls_last_30,total_complaints,customer_issues,financial_issues,pricing_mentions,avg_cc_sentiment,repeat_call_ratio,high_call_volume,prospect_outcome
0,AA0794,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
1,AA0794,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
2,AA0794,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
3,AA0794,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
4,AA0794,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won


**Cell 2: Create Churn Target**  
This cell converts the prospect outcome into a binary target where 1 means churned and 0 means retained.  
- **Purpose**: Define the dependent variable used in all subsequent hypothesis tests.  
- **No hypothesis testing here**: this is target creation.  
- **Expected output**: counts of churn vs. non-churn customers.

In [16]:
df["target"] = (df["prospect_outcome"] == "Churned").astype(int)

df["target"].value_counts()

target
0    102310
1     15011
Name: count, dtype: int64

**Cell 3: Define Feature Lists**  
This cell lists the numerical CC calls features and categorical features to test against churn.  
- **Purpose**: Specify which variables will be evaluated in hypothesis testing.  
- **No hypothesis testing here**: feature selection only.  
- **Expected output**: feature lists stored for later tests.

In [17]:
num_cols = [
    "total_cc_calls",
    "days_since_last_cc_call",
    "cc_calls_last_7",
    "cc_calls_last_30",
    "total_complaints",
    "customer_issues",
    "financial_issues",
    "pricing_mentions",
    "avg_cc_sentiment",
    "repeat_call_ratio",
    "high_call_volume"
]

cat_cols = ["has_cc_calls_data"]

**Cell 4: Numerical Hypothesis Testing**  
This cell performs Mann-Whitney U tests for each numerical CC calls feature to compare churn and non-churn groups.  
- **Null Hypothesis (H₀)**: The distribution of the feature is the same for churn and non-churn customers.  
- **Alternative Hypothesis (H₁)**: The distribution of the feature differs between churn and non-churn customers.  
- **Test used**: Mann-Whitney U test (non-parametric).  
- **Expected output**: summary table with p-values for all numerical features.

In [18]:
results = []

for col in num_cols:
    churn = df[df["target"] == 1][col].dropna()
    non_churn = df[df["target"] == 0][col].dropna()

    stat, p_value = mannwhitneyu(churn, non_churn)

    results.append({
        "feature": col,
        "p_value": p_value
    })

num_results = pd.DataFrame(results)
num_results

,feature,p_value
0,total_cc_calls,0.501542
1,days_since_last_cc_call,0.051867
2,cc_calls_last_7,0.426723
3,cc_calls_last_30,0.144910
4,total_complaints,1.000000
5,customer_issues,1.000000
6,financial_issues,1.000000
7,pricing_mentions,0.270168
8,avg_cc_sentiment,0.000003
9,repeat_call_ratio,0.466753


**Cell 5: Adjust for Multiple Testing**  
This cell applies Benjamini-Hochberg correction to the numerical feature p-values.  
- **Purpose**: Reduce false discovery rate when many features are tested.  
- **Key output**: `adjusted_p` and `significant` flags.  
- **Decision rule**: feature is significant if `adjusted_p < 0.05`.

In [19]:
p_values = num_results["p_value"]

adjusted_p = multipletests(p_values, method="fdr_bh")[1]

num_results["adjusted_p"] = adjusted_p
num_results["significant"] = adjusted_p < 0.05

num_results = num_results.sort_values("adjusted_p")

num_results

,feature,p_value,adjusted_p,significant
8,avg_cc_sentiment,0.000003,0.000033,True
1,days_since_last_cc_call,0.051867,0.285266,False
3,cc_calls_last_30,0.144910,0.531335,False
7,pricing_mentions,0.270168,0.742961,False
2,cc_calls_last_7,0.426723,0.788137,False
0,total_cc_calls,0.501542,0.788137,False
9,repeat_call_ratio,0.466753,0.788137,False
4,total_complaints,1.000000,1.000000,False
6,financial_issues,1.000000,1.000000,False
5,customer_issues,1.000000,1.000000,False


**Cell 6: Categorical Hypothesis Testing**  
This cell performs chi-square tests for categorical CC calls features, if any exist.  
- **Null Hypothesis (H₀)**: The categorical feature is independent of churn.  
- **Alternative Hypothesis (H₁)**: The categorical feature is associated with churn.  
- **Test used**: Chi-square test of independence.  
- **Expected output**: p-values per categorical feature, or an empty table if no categorical features are available.

In [20]:
cat_results = []

for col in cat_cols:
    table = pd.crosstab(df[col], df["target"])
    chi2, p_value, _, _ = chi2_contingency(table)

    cat_results.append({
        "feature": col,
        "p_value": p_value
    })

cat_results = pd.DataFrame(cat_results)
cat_results

,feature,p_value
0,has_cc_calls_data,3.123828e-303


**Cell 7: Save Test Results**  
This cell exports the numerical and categorical hypothesis test results to CSV files.  
- **Purpose**: Persist the final analysis output for reporting and future use.  
- **No hypothesis testing here**: this is the export step.  
- **Expected output**: saved result files under `reports/`.

In [21]:
num_results.to_csv(
    "../../reports/cc_calls_hypothesis_results.csv",
    index=False
)

cat_results.to_csv(
    "../../reports/cc_calls_categorical_results.csv",
    index=False
)